# Production-Grade Rate Limiting and Retry Patterns for the Anthropic API

Real applications hitting the Anthropic API will eventually encounter:
- **429 RateLimitError** — too many requests in a time window
- **500/529 InternalServerError** — transient server-side failures
- **Timeout / connection errors** — network blips

Naive retry logic either thundering-herds the API (making the problem worse) or gives up too early. This notebook walks through production-proven patterns:

| Pattern | Problem it solves |
|---|---|
| Exponential backoff + jitter | Avoids thundering herd on retry |
| Token budget tracker | Proactive quota management |
| Rate-limit headers | Real-time remaining capacity |
| Async semaphore | Controlled batch parallelism |

All patterns use `claude-haiku-4-5` — fast and cost-efficient for high-throughput use cases.

## 1. Setup

Install the required packages. `tenacity` is a battle-tested Python retry library that handles backoff strategies declaratively.

In [ ]:
%pip install anthropic tenacity --quiet

In [ ]:
import anthropic
import asyncio
import random
import time
from typing import Any

from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
    wait_random,
    before_sleep_log,
)
import logging

# Set up logging so tenacity prints retry attempts
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# The client automatically reads ANTHROPIC_API_KEY from the environment.
# Never hardcode keys in source code.
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5"

print("Setup complete. Using model:", MODEL)

## 2. The Naive Problem: Why Simple Retry Hurts

The most tempting approach is a tight retry loop:

```python
# DO NOT RUN — this pattern is intentionally broken
for attempt in range(5):
    try:
        response = client.messages.create(...)
        break
    except anthropic.RateLimitError:
        time.sleep(1)   # Fixed 1-second delay
        continue
```

**Why this is dangerous:**

1. **Fixed delay = thundering herd.** If 100 clients all hit a 429 at the same moment and all sleep for exactly 1 second, they all hammer the API again at the same moment — amplifying the spike instead of spreading it out.

2. **No jitter = synchronized retries.** Even with exponential backoff, if every instance of your service starts at the same time, they stay synchronized across retries.

3. **No differentiation between error types.** A 400 (bad request) should NOT be retried at all — it will never succeed. A 429 or 529 should be retried with backoff. A 401 (auth error) should fail fast.

The cell below is shown for educational purposes only — it is **NOT** executed.

In [ ]:
# DO NOT RUN — naive retry anti-pattern (shown for contrast only)

def naive_retry_bad_example(prompt: str) -> str:
    """Anti-pattern: fixed-delay retry with no jitter or error discrimination."""
    for attempt in range(5):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=256,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.content[0].text
        except Exception:          # catches EVERYTHING including 400s — wrong!
            time.sleep(1)          # fixed delay — thundering herd risk
    raise RuntimeError("All retries exhausted")

# Problems:
# - Retries on 400 Bad Request (will never succeed)
# - Fixed 1s sleep causes synchronized retries across distributed services
# - No logging of retry attempts
# - No upper bound on total wait time
print("Anti-pattern defined (not called).")

## 3. Exponential Backoff with Jitter Using `tenacity`

`tenacity` lets you express retry policy as a decorator, keeping business logic clean.

**Strategy breakdown:**
- `wait_exponential(multiplier=1, min=4, max=60)` — doubles the wait each attempt: 4s → 8s → 16s → 32s → 60s (capped)
- `+ wait_random(0, 2)` — adds 0–2 seconds of jitter to desynchronize retries across clients
- `stop_after_attempt(5)` — never retry more than 5 times total
- `retry_if_exception_type(...)` — only retry transient errors (429, 5xx); let 400s and 401s fail fast

In [ ]:
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
    wait_random,
    before_sleep_log,
)

# Retry only on errors that are transient and worth retrying
RETRYABLE_ERRORS = (
    anthropic.RateLimitError,       # 429 — slow down
    anthropic.InternalServerError,  # 500/529 — server blip
    anthropic.APIConnectionError,   # network timeout / connection drop
)

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=60) + wait_random(0, 2),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(RETRYABLE_ERRORS),
    before_sleep=before_sleep_log(logger, logging.WARNING),
    reraise=True,  # re-raise the original exception if all attempts fail
)
def _create_with_retry(**kwargs) -> anthropic.types.Message:
    """Internal function decorated with tenacity retry logic."""
    return client.messages.create(**kwargs)


def call_with_retry(
    client: anthropic.Anthropic,
    **kwargs: Any,
) -> anthropic.types.Message:
    """
    Drop-in wrapper around client.messages.create() with production-grade
    exponential backoff + jitter retry.

    Args:
        client: An anthropic.Anthropic instance.
        **kwargs: Passed directly to client.messages.create().

    Returns:
        anthropic.types.Message

    Raises:
        anthropic.BadRequestError: Immediately (not retried — bad input).
        anthropic.AuthenticationError: Immediately (fix your key).
        anthropic.RateLimitError: After 5 retries with backoff.
    """
    return _create_with_retry(**kwargs)


# --- Demo call ---
response = call_with_retry(
    client,
    model=MODEL,
    max_tokens=64,
    messages=[{"role": "user", "content": "Say 'retry works' in 5 words."}],
)
print(response.content[0].text)

# Example output:
# Retry works, confirmed successfully here.

## 4. Token Budget Tracker

Rate limits are enforced in two dimensions: **requests per minute (RPM)** and **tokens per minute (TPM)**. High-throughput apps often exhaust their token budget before their request budget.

A proactive token budget tracker:
- Sums `input_tokens + output_tokens` from every response
- Emits a warning when usage exceeds 80% of the assumed limit
- Sleeps proactively before the next call if the budget is nearly exhausted — preventing a 429 in the first place

Check your actual TPM limit in the [Anthropic Console → Limits](https://console.anthropic.com/settings/limits).

In [ ]:
import time
import warnings


class TokenBudgetTracker:
    """
    Tracks token usage over a rolling 60-second window.
    Raises a warning at 80% of assumed limit and sleeps proactively at 95%.

    Args:
        tokens_per_minute: Your account's TPM limit (check Anthropic Console).
        warn_threshold: Fraction of limit that triggers a warning (default 0.80).
        sleep_threshold: Fraction of limit that triggers a proactive sleep (default 0.95).
    """

    def __init__(
        self,
        tokens_per_minute: int = 40_000,
        warn_threshold: float = 0.80,
        sleep_threshold: float = 0.95,
    ) -> None:
        self.tokens_per_minute = tokens_per_minute
        self.warn_threshold = warn_threshold
        self.sleep_threshold = sleep_threshold
        self._window_start: float = time.monotonic()
        self._tokens_used: int = 0

    def _reset_if_needed(self) -> None:
        """Reset the counter if the 60-second window has elapsed."""
        elapsed = time.monotonic() - self._window_start
        if elapsed >= 60.0:
            self._window_start = time.monotonic()
            self._tokens_used = 0

    def record(self, message: anthropic.types.Message) -> None:
        """
        Record token usage from a completed API response.
        Call this immediately after every successful client.messages.create().
        """
        self._reset_if_needed()
        used = message.usage.input_tokens + message.usage.output_tokens
        self._tokens_used += used

        utilization = self._tokens_used / self.tokens_per_minute

        if utilization >= self.sleep_threshold:
            remaining_window = 60.0 - (time.monotonic() - self._window_start)
            sleep_secs = max(0.0, remaining_window)
            print(
                f"[TokenBudget] {utilization:.0%} of TPM limit reached. "
                f"Sleeping {sleep_secs:.1f}s to reset window."
            )
            time.sleep(sleep_secs)
            self._reset_if_needed()
        elif utilization >= self.warn_threshold:
            warnings.warn(
                f"[TokenBudget] {utilization:.0%} of TPM limit used "
                f"({self._tokens_used}/{self.tokens_per_minute} tokens). "
                "Consider slowing down.",
                stacklevel=2,
            )

    @property
    def current_usage(self) -> dict:
        """Return current window stats as a dict."""
        self._reset_if_needed()
        return {
            "tokens_used": self._tokens_used,
            "tokens_limit": self.tokens_per_minute,
            "utilization_pct": round(100 * self._tokens_used / self.tokens_per_minute, 1),
            "window_elapsed_s": round(time.monotonic() - self._window_start, 1),
        }


# --- Demo usage ---
tracker = TokenBudgetTracker(tokens_per_minute=40_000)

response = call_with_retry(
    client,
    model=MODEL,
    max_tokens=128,
    messages=[{"role": "user", "content": "Name three cloud providers."}],
)
tracker.record(response)

print(response.content[0].text)
print("\nBudget status:", tracker.current_usage)

# Example output:
# AWS, Google Cloud, and Microsoft Azure.
#
# Budget status: {'tokens_used': 47, 'tokens_limit': 40000, 'utilization_pct': 0.1, 'window_elapsed_s': 0.2}

## 5. Reading Rate-Limit Headers

The Anthropic API returns rate-limit information in HTTP response headers after every call. Reading these lets your app react to the *actual* remaining capacity rather than guessing from token counts.

Key headers:

| Header | Meaning |
|---|---|
| `x-ratelimit-limit-requests` | Your RPM cap |
| `x-ratelimit-remaining-requests` | Requests left in current window |
| `x-ratelimit-reset-requests` | ISO-8601 timestamp when RPM window resets |
| `x-ratelimit-limit-tokens` | Your TPM cap |
| `x-ratelimit-remaining-tokens` | Tokens left in current window |
| `x-ratelimit-reset-tokens` | ISO-8601 timestamp when TPM window resets |

Use `client.messages.with_raw_response.create()` to access the raw `httpx.Response` and its headers.

In [ ]:
def call_and_show_headers(prompt: str) -> anthropic.types.Message:
    """
    Make an API call and print the rate-limit headers from the response.
    Uses with_raw_response to access HTTP headers before the SDK parses the body.
    """
    with client.messages.with_raw_response.create(
        model=MODEL,
        max_tokens=64,
        messages=[{"role": "user", "content": prompt}],
    ) as raw:
        headers = raw.headers

        print("=== Rate Limit Headers ===")
        rate_limit_keys = [
            "x-ratelimit-limit-requests",
            "x-ratelimit-remaining-requests",
            "x-ratelimit-reset-requests",
            "x-ratelimit-limit-tokens",
            "x-ratelimit-remaining-tokens",
            "x-ratelimit-reset-tokens",
        ]
        for key in rate_limit_keys:
            value = headers.get(key, "(not present)")
            print(f"  {key}: {value}")

        remaining_requests = int(headers.get("x-ratelimit-remaining-requests", -1))
        remaining_tokens = int(headers.get("x-ratelimit-remaining-tokens", -1))

        if remaining_requests != -1 and remaining_requests < 5:
            print(f"\n[WARN] Only {remaining_requests} requests remaining in window!")
        if remaining_tokens != -1 and remaining_tokens < 1000:
            print(f"[WARN] Only {remaining_tokens} tokens remaining in window!")

        # Parse the response body from the raw response
        message = raw.parse()

    return message


msg = call_and_show_headers("What is 2 + 2?")
print("\nResponse:", msg.content[0].text)

# Example output:
# === Rate Limit Headers ===
#   x-ratelimit-limit-requests: 50
#   x-ratelimit-remaining-requests: 49
#   x-ratelimit-reset-requests: 2025-06-21T10:01:00Z
#   x-ratelimit-limit-tokens: 40000
#   x-ratelimit-remaining-tokens: 39947
#   x-ratelimit-reset-tokens: 2025-06-21T10:01:00Z
#
# Response: 4

## 6. Adaptive Throttle Using Headers

Instead of reacting to 429s after the fact, we can *proactively* slow down based on the `x-ratelimit-remaining-*` headers. The function below builds a small adaptive throttle: if remaining requests drops below 20% of the limit, it sleeps until the reset timestamp.

In [ ]:
from datetime import datetime, timezone


def adaptive_throttled_call(
    client: anthropic.Anthropic,
    low_water_fraction: float = 0.20,
    **kwargs: Any,
) -> anthropic.types.Message:
    """
    Makes an API call and proactively sleeps if remaining capacity is low.

    Args:
        client: anthropic.Anthropic instance.
        low_water_fraction: Sleep when remaining/limit < this fraction (default 0.20).
        **kwargs: Passed to messages.create().

    Returns:
        anthropic.types.Message
    """
    with client.messages.with_raw_response.create(**kwargs) as raw:
        h = raw.headers

        limit_req = int(h.get("x-ratelimit-limit-requests", 0) or 0)
        remaining_req = int(h.get("x-ratelimit-remaining-requests", 0) or 0)
        reset_req = h.get("x-ratelimit-reset-requests", "")

        limit_tok = int(h.get("x-ratelimit-limit-tokens", 0) or 0)
        remaining_tok = int(h.get("x-ratelimit-remaining-tokens", 0) or 0)
        reset_tok = h.get("x-ratelimit-reset-tokens", "")

        def _sleep_until(reset_ts: str, label: str) -> None:
            if not reset_ts:
                return
            try:
                reset_dt = datetime.fromisoformat(reset_ts.replace("Z", "+00:00"))
                now = datetime.now(timezone.utc)
                secs = (reset_dt - now).total_seconds()
                if secs > 0:
                    print(f"[AdaptiveThrottle] {label} low — sleeping {secs:.1f}s until reset.")
                    time.sleep(secs)
            except ValueError:
                pass  # malformed timestamp; skip sleep

        if limit_req > 0 and (remaining_req / limit_req) < low_water_fraction:
            _sleep_until(reset_req, f"requests ({remaining_req}/{limit_req})")

        if limit_tok > 0 and (remaining_tok / limit_tok) < low_water_fraction:
            _sleep_until(reset_tok, f"tokens ({remaining_tok}/{limit_tok})")

        return raw.parse()


# --- Demo ---
msg = adaptive_throttled_call(
    client,
    model=MODEL,
    max_tokens=64,
    messages=[{"role": "user", "content": "Name the Python creator."}],
)
print(msg.content[0].text)

# Example output:
# Guido van Rossum.

## 7. Batch-Friendly Retry with Async Semaphore

When processing a list of items (e.g., classifying 200 support tickets), you want:
- **Parallelism** — run multiple requests concurrently, not one-by-one
- **Bounded concurrency** — not 200 simultaneous requests (that will definitely 429)
- **Per-item retry** — if one item fails, retry it independently without failing the batch

`asyncio.gather` + `asyncio.Semaphore` is the idiomatic solution. The semaphore acts as a gate: at most `max_concurrent` coroutines can be inside the API call at any time.

In [ ]:
import asyncio
from dataclasses import dataclass, field


@dataclass
class BatchResult:
    """Result for a single item in the batch."""
    item: str
    response: str = ""
    error: str = ""
    attempts: int = 0


async def _process_one(
    item: str,
    client: anthropic.AsyncAnthropic,
    semaphore: asyncio.Semaphore,
    max_retries: int = 4,
) -> BatchResult:
    """Process a single item with exponential backoff inside a semaphore gate."""
    result = BatchResult(item=item)

    for attempt in range(1, max_retries + 1):
        result.attempts = attempt
        try:
            async with semaphore:
                msg = await client.messages.create(
                    model=MODEL,
                    max_tokens=128,
                    messages=[{"role": "user", "content": item}],
                )
            result.response = msg.content[0].text
            return result
        except (anthropic.RateLimitError, anthropic.InternalServerError) as exc:
            if attempt == max_retries:
                result.error = f"{type(exc).__name__}: {exc}"
                return result
            # Exponential backoff with jitter
            base_delay = min(4 * (2 ** (attempt - 1)), 60)
            jitter = random.uniform(0, 2)
            delay = base_delay + jitter
            print(f"[Batch] Item={item!r} attempt={attempt} → retrying in {delay:.1f}s")
            await asyncio.sleep(delay)
        except anthropic.BadRequestError as exc:
            # Do not retry bad inputs — they will never succeed
            result.error = f"BadRequest (not retried): {exc}"
            return result

    return result  # unreachable, but satisfies type checker


async def process_batch(
    items: list[str],
    client: anthropic.AsyncAnthropic,
    max_concurrent: int = 3,
) -> list[BatchResult]:
    """
    Process a list of prompts concurrently, capped at max_concurrent in-flight requests.

    Args:
        items: List of prompt strings to send to the API.
        client: anthropic.AsyncAnthropic instance.
        max_concurrent: Maximum simultaneous API calls (default 3).

    Returns:
        List of BatchResult, one per item, in the same order as input.
    """
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [_process_one(item, client, semaphore) for item in items]
    return await asyncio.gather(*tasks)


print("Async batch functions defined.")

In [ ]:
# Run the async batch processor
# In a Jupyter notebook, the event loop is already running,
# so we use asyncio.run() or await directly depending on the kernel.

async_client = anthropic.AsyncAnthropic()

prompts = [
    "What is the capital of France?",
    "What is 7 * 8?",
    "Name one programming language created after 2010.",
    "What does HTTP stand for?",
    "Who wrote the novel '1984'?",
]

results = await process_batch(prompts, async_client, max_concurrent=3)

for r in results:
    status = "OK" if not r.error else "ERROR"
    print(f"[{status}] (attempts={r.attempts}) {r.item!r}")
    if r.response:
        print(f"  -> {r.response.strip()}")
    if r.error:
        print(f"  -> {r.error}")

# Example output:
# [OK] (attempts=1) 'What is the capital of France?'
#   -> Paris.
# [OK] (attempts=1) 'What is 7 * 8?'
#   -> 56
# [OK] (attempts=1) 'Name one programming language created after 2010.'
#   -> Rust (released in 2010, stabilized in 2015).
# [OK] (attempts=1) 'What does HTTP stand for?'
#   -> HyperText Transfer Protocol.
# [OK] (attempts=1) 'Who wrote the novel \'1984\'?'
#   -> George Orwell.

## 8. Putting It All Together

In a real production pipeline, you'd compose these patterns:
1. **Async batch** for throughput (Sec 7)
2. **Per-call retry** for transient errors (Sec 3)
3. **Token budget tracking** to manage quota proactively (Sec 4)
4. **Header inspection** to react to real-time server signals (Sec 5–6)

Below is a skeleton showing how all four layers nest together.

In [ ]:
class ProductionAPIClient:
    """
    Production-grade Anthropic API client combining:
    - Tenacity exponential backoff + jitter retry
    - Token budget tracking with proactive sleep
    - Rate-limit header inspection
    - Async batch processing with semaphore
    """

    def __init__(
        self,
        tokens_per_minute: int = 40_000,
        max_concurrent: int = 3,
    ) -> None:
        self.sync_client = anthropic.Anthropic()
        self.async_client = anthropic.AsyncAnthropic()
        self.tracker = TokenBudgetTracker(tokens_per_minute=tokens_per_minute)
        self.max_concurrent = max_concurrent

    def call(self, **kwargs: Any) -> anthropic.types.Message:
        """Single synchronous call with retry and budget tracking."""
        response = call_with_retry(self.sync_client, **kwargs)
        self.tracker.record(response)
        return response

    async def batch(self, items: list[str], **shared_kwargs: Any) -> list[BatchResult]:
        """Process a list of prompts with async concurrency and per-item retry."""
        # Build full prompt dicts from items + shared kwargs (model, max_tokens, etc.)
        return await process_batch(items, self.async_client, self.max_concurrent)


# --- Demo ---
prod_client = ProductionAPIClient(tokens_per_minute=40_000, max_concurrent=3)

msg = prod_client.call(
    model=MODEL,
    max_tokens=64,
    messages=[{"role": "user", "content": "What is the speed of light?"}],
)
print(msg.content[0].text)
print("Token budget:", prod_client.tracker.current_usage)

# Example output:
# Approximately 299,792,458 meters per second (about 3 × 10^8 m/s) in a vacuum.
#
# Token budget: {'tokens_used': 71, 'tokens_limit': 40000, 'utilization_pct': 0.2, 'window_elapsed_s': 1.1}

## 9. Summary: When to Use Each Pattern

| Pattern | Use When | Code Location |
|---|---|---|
| **Tenacity exponential backoff** | Any production app — wraps every API call | `call_with_retry()` (Sec 3) |
| **Jitter on backoff** | Multiple service instances calling the same API | `wait_random(0, 2)` in tenacity decorator |
| **Token budget tracker** | High-volume apps; TPM limit is the binding constraint | `TokenBudgetTracker` (Sec 4) |
| **Rate-limit header inspection** | When you need real-time capacity signals from the server | `with_raw_response.create()` (Sec 5) |
| **Adaptive throttle** | Long-running jobs where spilling into next window is acceptable | `adaptive_throttled_call()` (Sec 6) |
| **Async semaphore batch** | Processing lists of 10+ items; parallelism > 1 is needed | `process_batch()` (Sec 7) |

### Key Principles Recap

1. **Never retry 400 Bad Request** — bad input will always be bad. Fix the prompt.
2. **Always use jitter** — even `random.uniform(0, 2)` breaks synchronized retry storms.
3. **Track tokens, not just requests** — TPM limits often bite before RPM limits do.
4. **Prefer proactive throttling** — sleeping before a 429 is cheaper than recovering from one.
5. **Use `asyncio.Semaphore` for batches** — it's the simplest correct tool for bounded concurrency.
6. **Read the headers** — the API tells you exactly how much budget remains; use that information.

### Further Reading

- [Anthropic API Rate Limits](https://docs.anthropic.com/en/api/rate-limits)
- [tenacity documentation](https://tenacity.readthedocs.io/)
- [Python asyncio Semaphore](https://docs.python.org/3/library/asyncio-sync.html#asyncio.Semaphore)